<a href="https://colab.research.google.com/github/Eswar-8/Premiumization-Markers/blob/main/Premiumization-Markers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ydata-profiling

In [ ]:
from google.colab import auth
from google.cloud import bigquery
from ydata_profiling import ProfileReport
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.metrics import mean_absolute_error, r2_score

# Authentication and Data Collection

In [ ]:
auth.authenticate_user()
print('Authenticated')

In [ ]:
client = bigquery.Client(project='alert-study-493302-r4')

In [ ]:
def sales(year):
  query= f"SELECT * FROM `bigquery-public-data.iowa_liquor_sales.sales` WHERE EXTRACT(YEAR FROM date) = {year}"
  df=client.query(query).to_dataframe()
  df['zip_code']=df['zip_code'].astype(str).str.split('.').str[0]
  return df

def census(year):
  query=f"""SELECT geo_id as zip_code, total_pop,households,median_age,median_income,
  SAFE_DIVIDE((bachelors_degree+masters_degree+graduate_professional_degree),total_pop) as edu_index,
  SAFE_DIVIDE(employed_pop,(employed_pop+unemployed_pop)) as employment_rate
  From `bigquery-public-data.census_bureau_acs.zip_codes_{year}_5yr` """
  census_data=client.query(query).to_dataframe()
  return census_data

In [ ]:
sales_2017=sales(2017)
sales_2018=sales(2018)

census_2017=census(2017)
census_2018=census(2018)

In [ ]:
train_df=sales_2017.merge(census_2017,on='zip_code',how='left')
test_df=sales_2018.merge(census_2018,on='zip_code',how='left')

In [ ]:
print(len(train_df))
print(len(test_df))

In [ ]:
profile=ProfileReport(train_df,title="Before Data Cleaning")
profile.to_notebook_iframe()
profile.to_file('Before Data Cleaning.html')

In [ ]:
train_df.isnull().sum()

#Data Cleaning

In [ ]:
missing=train_df[train_df['zip_code'].isnull() | train_df['city'].isnull() | train_df['county'].isnull()]
print(len(missing))

In [ ]:
def preprocess(df):

  drop_columns = ['store_number','store_location','store_name','address',
                  'county_number','vendor_number','category',]
  df = df.drop(columns=drop_columns)

  df=df.dropna(subset=['city'])

  columns=['total_pop','households','median_age','median_income','edu_index','employment_rate']
  for a in columns:
    df[a]=df[a].fillna(df[a].median())

  df['category_name']=df['category_name'].fillna('UNKNOWN')

  prime_categories={
      'TENNESSEE WHISKIES': 'WHISKY',
      'SINGLE BARREL BOURBON WHISKIES': 'WHISKY',
      'STRAIGHT BOURBON WHISKIES': 'WHISKY',
      'CANADIAN WHISKIES': 'WHISKY',
      'IRISH WHISKIES': 'WHISKY',
      'BLENDED WHISKIES': 'WHISKY',
      'STRAIGHT RYE WHISKIES': 'WHISKY',
      'SINGLE MALT SCOTCH': 'WHISKY',
      'SCOTCH WHISKIES': 'WHISKY',
      'CORN WHISKIES': 'WHISKY',
      'BOTTLED IN BOND BOURBON': 'WHISKY',
      'IOWA DISTILLERY WHISKIES': 'WHISKY',
      'WHISKEY LIQUEUR': 'WHISKY',

      'AMERICAN VODKAS': 'VODKA',
      'IMPORTED VODKAS': 'VODKA',
      'AMERICAN FLAVORED VODKA': 'VODKA',
      'IMPORTED FLAVORED VODKA': 'VODKA',

      'MIXTO TEQUILA': 'TEQUILA',
      '100% AGAVE TEQUILA': 'TEQUILA',
      'MEZCAL': 'TEQUILA',

      'SPICED RUM': 'RUM',
      'FLAVORED RUM': 'RUM',
      'WHITE RUM': 'RUM',
      'GOLD RUM': 'RUM',
      'AGED DARK RUM': 'RUM',

      'AMERICAN DRY GINS': 'GIN',
      'AMERICAN SLOE GINS': 'GIN',
      'IMPORTED DRY GINS': 'GIN',
      'FLAVORED GIN': 'GIN',

      'AMERICAN BRANDIES': 'BRANDY',
      'IMPORTED BRANDIES': 'BRANDY',

      'AMERICAN SCHNAPPS': 'LIQUEUR',
      'IMPORTED SCHNAPPS': 'LIQUEUR',
      'AMERICAN CORDIALS & LIQUEURS': 'LIQUEUR',
      'IMPORTED CORDIALS & LIQUEURS': 'LIQUEUR',
      'CREAM LIQUEURS': 'LIQUEUR',
      'COFFEE LIQUEURS': 'LIQUEUR',
      'TRIPLE SEC': 'LIQUEUR',

      'UNKNOWN':'OTHER'
  }
  df['primary_category']=df['category_name'].map(prime_categories)
  df['primary_category']=df['primary_category'].fillna('OTHER')
  return df

In [ ]:
final_train_df=preprocess(train_df)
final_test_df=preprocess(test_df)

In [ ]:
final_test_df=final_test_df.dropna()

In [ ]:
# from ydata_profiling import ProfileReport

prof=ProfileReport(final_train_df,title="After Data Cleaning")
prof.to_notebook_iframe()
prof.to_file('After Data Cleaning.html')

#Feature Engineering

In [ ]:
def feaengineering(df):
  df['price_per_ml'] = df['state_bottle_retail']/df['bottle_volume_ml']

  countylevel = df.groupby('zip_code')['price_per_ml'].agg(['mean','std']).reset_index()
  countylevel.columns = ['zip_code','county_mean','county_std']
  df=df.merge(countylevel,on='zip_code',how='left')
  df['z_score']=(df['price_per_ml']-df['county_mean'])/(df['county_std'] + 1e-10)

  segment_dummies = pd.get_dummies(df['primary_category'], prefix='segment')
  df = pd.concat([df, segment_dummies], axis=1)

  return df

In [ ]:
final_train_df=feaengineering(final_train_df)
final_test_df=feaengineering(final_test_df)

In [ ]:
plt.figure(figsize=(12,6))

sns.histplot(final_train_df['z_score'],kde=True,bins=50,color='red')
plt.show()

In [ ]:
target='z_score'
features = [
           'sale_dollars',
            'volume_sold_liters',
            'median_income',
            'total_pop',
            'state_bottle_retail',
           'county_mean'
            ]

Model                | MAE        | R2 Score  
---------------------------------------------
Random Forest        | 0.1578     | 0.8116    
Ridge Regression    | 0.3643     | 0.2894    
Bayesian Ridge      | 0.3644     | 0.2891  

In [ ]:
target='z_score'
features = [
           'sale_dollars',
            'volume_sold_liters',
            'median_income',
            'total_pop',
            'state_bottle_retail',
           'county_mean',
           'county_std'
            ]

Model                | MAE        | R2 Score  
---------------------------------------------
Random Forest        | 0.0779     | 0.9039    
Ridge Regression    | 0.3748     | 0.2834    
Bayesian Ridge      | 0.3736     | 0.2841  

In [ ]:
target='z_score'
features = ['sale_dollars',
            'volume_sold_liters',
            'median_income',
            'total_pop',
            'state_bottle_retail',
           'county_mean',
           'county_std','median_age','segment_BRANDY', 'segment_GIN', 'segment_LIQUEUR', 'segment_OTHER',
       'segment_RUM', 'segment_TEQUILA', 'segment_VODKA', 'segment_WHISKY'
            ]

Model                | MAE        | R2 Score  
---------------------------------------------
Random Forest        | 0.0815     | 0.9014    
Ridge Regression    | 0.3860     | 0.2970    
Bayesian Ridge      | 0.3846     | 0.2981   

In [ ]:
target='z_score'
features = [

            'volume_sold_liters',
            'median_income',
            'total_pop',
            'county_mean',
            'price_per_ml',
            'edu_index',
    'employment_rate',
    'median_age','segment_BRANDY', 'segment_GIN', 'segment_LIQUEUR', 'segment_OTHER',
       'segment_RUM', 'segment_TEQUILA', 'segment_VODKA', 'segment_WHISKY'
            ]

Model | MAE | R2 Score



Random Forest | 0.2301 | 0.7163

Ridge Regression  | 0.1929 | 0.6462

Bayesian Ridge | 0.1935 | 0.6431

# Model Training

In [ ]:
# 1. Prepare Data
X_train, y_train = final_train_df[features], final_train_df[target]
X_test, y_test = final_test_df[features], final_test_df[target]

# 2. Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# 3. Linear Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_preds = ridge_model.predict(X_test)

# 4. Bayesian Ridge
bayesian_model = BayesianRidge()
bayesian_model.fit(X_train, y_train)
bay_preds = bayesian_model.predict(X_test)

# --- Comparison Report ---
models = {
    "Random Forest": rf_preds,
    "Ridge Regression": ridge_preds,
    "Bayesian Ridge": bay_preds
}

print(f"{'Model':<20} | {'MAE':<10} | {'R2 Score':<10}")
print("-" * 45)

for name, preds in models.items():
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name:<20} | {mae:<10.4f} | {r2:<10.4f}")

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features)
importances.nlargest(10).plot(kind='barh')
plt.title('Top 10 Features Driving Z-Score')
plt.show()

In [ ]:
results=pd.DataFrame({'zip_code': final_test_df['zip_code'],
                      'actual_premium': y_test,
                      'predicted_premium': rf_preds

})

results['oportunity_gap']=results['predicted_premium']-results['actual_premium']

In [ ]:
results[(results['actual_premium']>0.5) | (results['predicted_premium']>0.5)]

In [ ]:
premium_zip_summary = results.groupby('zip_code').agg(
    count=('actual_premium', 'count'),
    avg_actual=('actual_premium', lambda x: x.quantile(0.9)),
    max_actual=('actual_premium', 'max'),
    avg_gap=('oportunity_gap', 'sum')
).reset_index()


In [ ]:
premium_zip_summary.sort_values(by='avg_gap', ascending=False)